# Capstone 03: PolicyPal Starter
**PolicyPal: Enterprise HR and IT Policy Chatbot**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
This notebook generates the project data, shows a working baseline, and seeds your golden set. Beat the baseline in your project.

* **Option A (Google Colab)** cells are active; **Option B (VS Code)** lines are commented. Toggle with `Ctrl + /`.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai scikit-learn numpy pandas rank_bm25

# ===== Option B: VS Code / local =====
# pip install openai scikit-learn numpy pandas rank_bm25
# Create a .env file or set the variable in your terminal:  OPENAI_API_KEY=sk-...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
"""
import json, os, re
import numpy as np

def _get_key():
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")                 # change to any current chat model you have access to
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None):
    if ONLINE:
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}])
        return r.choices[0].message.content
    return offline_llm(prompt, system)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, ONLINE

## 2. Sample policy documents

In [ ]:
%%writefile policies.md
# Sakura Logistics Employee Handbook (sample)

## 1. Working hours
Standard working hours are 9:30 to 18:30, Monday to Friday. Teams in the Tokyo office follow JST and teams in the Bengaluru office follow IST.
Flexible start times between 8:00 and 11:00 are allowed with manager approval.

## 2. Leave policy
Full-time employees receive 18 days of paid leave per year. Leave accrues monthly at 1.5 days.
Unused leave can be carried forward up to a maximum of 10 days. Carried forward leave expires on 31 March of the following year.
Sick leave is separate: employees receive 8 days of paid sick leave per year. A medical certificate is required for sick leave longer than 2 consecutive days.

## 3. Remote work
Employees may work remotely up to 2 days per week. Remote work must be logged in the HR portal by 10:00 on the same day.
Remote work from another country requires approval from HR and the security team at least 14 days in advance.

## 4. Expenses
Travel expenses must be submitted within 30 days with receipts. Meals during business travel are reimbursed up to 2,000 INR or 4,000 JPY per day.
Expenses above 50,000 INR require approval from a director before booking.

## 5. Security
Laptops must use full disk encryption. Customer data must never be copied to personal devices or uploaded to unapproved AI tools.
Security incidents must be reported to security@sakura-logistics.example within 1 hour of discovery.

## IT FAQ
Error VPN-417 means your VPN certificate has expired. Renew it from the IT self-service portal and restart the VPN client.
Password resets are done through the self-service portal. Passwords must be at least 14 characters long.
Approved AI tools are listed on the IT intranet page. Do not paste customer data into tools that are not on that list.
New laptops are issued within 5 working days of a request approved by your manager.

## Travel FAQ
Book flights through the corporate travel desk at least 10 days before travel. Economy class is standard for flights under 6 hours.
Hotel stays are reimbursed up to 6,000 INR per night in India and 15,000 JPY per night in Japan.
Visa support letters are issued by HR within 3 working days.

## Visa support
Visa support letters for Japan are issued by HR within 3 working days. Business visas need a passport, an invitation letter and the HR support letter.

## 3. Baseline: chunk, retrieve, answer with citations, refuse when unsure

In [ ]:
import re, json, numpy as np
text = open("policies.md").read()
chunks = []
for sec in re.split(r"\n(?=## )", text):
    lines = sec.strip().splitlines()
    if lines and lines[0].startswith("## "):
        title = lines[0][3:]
        for s in re.split(r"(?<=[.!?])\s+", " ".join(lines[1:])):
            if s.strip(): chunks.append({"id": f"c{len(chunks) + 1}", "section": title, "text": s.strip()})
V = embed([f"{c['section']}: {c['text']}" for c in chunks])
RULES = "Answer using ONLY the context and cite chunk ids like [c3]. If the answer is not there, say \"I don't know\"."
def ask(q, k=3, threshold=0.15):
    sims = V @ embed(q)[0]; top = np.argsort(sims)[::-1][:k]
    if sims[top[0]] < threshold:
        return "I don't know. I have created a ticket for the HR help desk.", []
    ctx = "\n".join(f"[{chunks[i]['id']}] {chunks[i]['text']}" for i in top)
    return llm(f"Context:\n{ctx}\n\nQuestion: {q}", system=RULES, temperature=0), [chunks[i]["id"] for i in top]
for q in ["When does carried forward leave expire?", "Can I use any AI tool with customer data?", "Is there a company gym?"]:
    a, ids = ask(q); print(f"Q: {q}\nA: {a}\n   {ids}\n")

## 4. Golden set seed

In [ ]:
golden = [{"id": "pol-001", "question": "When does carried forward leave expire?", "must_include": ["31 March"]},
          {"id": "pol-002", "question": "Can I use any AI tool with customer data?", "must_include": ["approved"]},
          {"id": "pol-003", "question": "Is there a company gym?", "must_include": ["don't know"]}]
with open("golden_seed.jsonl", "w") as f:
    for g in golden: f.write(json.dumps(g) + "\n")
print("Seeded", len(golden), "golden items. Extend to 30, including Japanese questions and injection attempts.")

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)